---
title: "Buildings & Urban Environment"
format:
  html:
    code-fold: true
execute:
  enabled: false
---

IPCC AR6 chapters, glossary terms, and acronyms related to buildings, architecture, and urban environments.

**Data sources:** `corpus-ar6.csv`, `glossary-ar6.csv`, `acronyms-ar6.csv`  
**Instructions:** Run all cells then re-render with Quarto to update outputs.

In [1]:
#| include: false
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

print("Libraries imported")

Libraries imported


## Configuration

CSV paths are relative to this notebook.

In [2]:
#| include: false
CORPUS_CSV   = "../data-xml-dtd/corpus-ar6.csv"
GLOSSARY_CSV = "../data-xml-dtd/glossary-ar6.csv"
ACRONYMS_CSV = "../data-xml-dtd/acronyms-ar6.csv"

print("Config set")

Config set


## Analysis

In [3]:
#| echo: false
# -- Load CSVs ----------------------------------------------------------------
corpus_df  = pd.read_csv(CORPUS_CSV, encoding='utf-8')
gloss_df   = pd.read_csv(GLOSSARY_CSV, encoding='utf-8').iloc[1:].reset_index(drop=True)
acronym_df = pd.read_csv(ACRONYMS_CSV, encoding='utf-8-sig')

def get_col(df, keyword):
    for c in df.columns:
        if keyword in c:
            return df[c].fillna('').astype(str)
    return pd.Series([''] * len(df))

corpus_title  = get_col(corpus_df, 'TITLE')
corpus_source = get_col(corpus_df, 'SOURCE').str.strip()
corpus_wiki   = get_col(corpus_df, 'WIKI').str.strip()
corpus_tags   = get_col(corpus_df, 'TAGLIST').str.strip()
corpus_desc   = get_col(corpus_df, 'DESCRIPTION').str.strip()

# -- Filter chapters related to buildings / urban / architecture --------------
CHAPTER_KW = ['building', 'urban', 'architect', 'city', 'cities', 'settlement']

mask = (
    corpus_title.str.lower().apply(lambda t: any(k in t for k in CHAPTER_KW)) |
    corpus_tags.str.lower().apply(lambda t: any(k in t for k in CHAPTER_KW))
)
chapters = corpus_df[mask].copy()
chapters['_title']  = corpus_title[mask].values
chapters['_source'] = corpus_source[mask].values
chapters['_wiki']   = corpus_wiki[mask].values
chapters['_desc']   = corpus_desc[mask].values

def get_wg(url):
    u = url.lower()
    if 'wg3' in u or 'wgiii' in u: return 'WGIII &mdash; Mitigation'
    if 'wg2' in u or 'wgii' in u:  return 'WGII &mdash; Impacts &amp; Adaptation'
    if 'wg1' in u or 'wgi' in u:   return 'WGI &mdash; Physical Science'
    return 'Cross-WG'

chapters['_wg'] = chapters['_wiki'].apply(get_wg)

# -- Chapter HTML table -------------------------------------------------------
rows_html = ''
for _, row in chapters.iterrows():
    src = row['_source']
    link = f'<a href="{src}" target="_blank">{row["_title"]}</a>' if src else row['_title']
    rows_html += f'<tr><td>{link}</td><td>{row["_wg"]}</td><td>{row["_desc"]}</td></tr>\n'

display(HTML(f'''
<h3>Chapters related to Buildings, Architecture &amp; Urban Environments ({len(chapters)})</h3>
<table class="table table-sm table-striped">
  <thead><tr><th>Chapter</th><th>Working Group</th><th>Report</th></tr></thead>
  <tbody>{rows_html}</tbody>
</table>
'''))

# -- Acronym count ------------------------------------------------------------
ACRONYM_KW = ['urban', 'building', 'city', 'cities', 'settlement',
               'infrastructure', 'architect', 'housing', 'retrofit']

acr_col  = 'Acronym'
desc_col = 'Description'

def acr_match(row):
    text = (str(row.get(acr_col, '')) + ' ' + str(row.get(desc_col, ''))).lower()
    return any(k in text for k in ACRONYM_KW)

building_acr = acronym_df[acronym_df.apply(acr_match, axis=1)]
acr_count    = building_acr[acr_col].nunique()

display(HTML(
    f'<p><strong>Acronyms related to buildings &amp; urban environments:</strong> {acr_count} unique terms</p>'
))

Chapter,Working Group,Report
"Cities, Settlements and Key Infrastructure",WGII — Impacts & Adaptation,"Working Group II: Climate Change 2022 – Impacts, Adaptation and Vulnerability"
Cities and Settlements by the Sea,WGII — Impacts & Adaptation,"Working Group II: Climate Change 2022 – Impacts, Adaptation and Vulnerability"
Urban Systems and Other Settlements,WGIII — Mitigation,Working Group III: Climate Change 2022 – Mitigation of Climate Change
Buildings,WGIII — Mitigation,Working Group III: Climate Change 2022 – Mitigation of Climate Change


In [4]:
#| echo: false
# -- Glossary word cloud (Plotly treemap) -------------------------------------
TERM_KW = ['urban', 'city', 'cities', 'settlement', 'building', 'architect',
           'megacity', 'peri-urban', 'housing', 'retrofit', 'infrastructure']

def term_wg_label(series_str):
    s = str(series_str).upper()
    has_ii  = 'WGII' in s
    has_iii = 'WGIII' in s
    if has_iii and has_ii: return 'WGII + WGIII'
    if has_iii: return 'WGIII'
    if has_ii:  return 'WGII'
    return 'Other'

# Filter glossary: term name must contain a keyword
term_rows = [
    row for _, row in gloss_df.iterrows()
    if any(k in str(row.get('Category', '')).lower() for k in TERM_KW)
]

groups = ['WGII', 'WGIII', 'WGII + WGIII', 'Other']
labels  = ['Built Environment'] + groups
parents = [''] + ['Built Environment'] * len(groups)
values  = [0] * len(labels)

for row in term_rows:
    term  = str(row.get('Category', ''))
    defn  = str(row.get('Definition (Description)', ''))
    wg    = term_wg_label(row.get('Series', ''))
    size  = max(len(defn) // 30, 1)
    labels.append(term)
    parents.append(wg)
    values.append(size)

fig = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    values=values,
    textinfo='label',
    marker=dict(colorscale='Teal'),
    hovertemplate='<b>%{label}</b><br>%{parent}<extra></extra>',
))
fig.update_layout(
    title=dict(
        text=f'Glossary Terms related to Buildings & Urban Environments ({len(term_rows)} terms)',
        font=dict(size=15), x=0.0),
    height=520,
    margin=dict(t=50, b=10, l=10, r=10),
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))